# XGBoost Adaptive Resource Allocation Model
### Bachelor Thesis — Hybrid DE-WOA Cluster Manager

**Goal:** Predict optimal fitness weights `(w_cpu, w_ram, w_io, w_energy)` and dynamic thresholds  
for **Warning**, **Critical**, and **Scale-Down** levels from live cluster metrics, with **incremental learning**  
so the model adapts every 5 minutes as new data arrives.

---

### Architecture Overview
```
Proxmox API + Prometheus  ──►  Feature Engineering  ──►  16 XGBRegressors  ──►  Predicted weights & thresholds
        ▲                                                        │
        │                                                        ▼
   poll_cluster()                                      ml_service.py  /sync
   (every 10s)                                                   │
        │                                                        ▼
   _record_ml_snapshot()                              main.py  _ml_config
   (pre-engineers features)                                      │
        │                                                        ▼
   metrics_latest.csv                                hybrid_de_woa_provisioning()
   logs_latest.csv                                   check_scaledown()
        │
        ▼
   incremental_learning.py  (every 5 min)
   xgb.train(xgb_model=existing)  →  updated .ubj files
        │
        ▼
   ml_service.py  /reload-models
```

### Data flow (training vs. production)

| Phase | Metrics source | Feature engineering |
|---|---|---|
| **Training** (this notebook) | `1_prometheus_raw_metrics.csv` — raw cumulative counters from Prometheus/node_exporter | `engineer_features()` computes deltas/rates |
| **Production** (incremental) | `metrics_latest.csv` written by `main.py`'s `_record_ml_snapshot()` | Features are **already computed** — `engineer_features()` is **not called** |

This asymmetry is intentional and documented in `incremental_learning.py`.

---

### 16 Prediction Targets
| Group | Targets |
|---|---|
| Fitness weights | `w_cpu`, `w_ram`, `w_io`, `w_energy` |
| Warning thresholds | `thresh_cpu_warn`, `thresh_ram_warn`, `thresh_disk_warn`, `thresh_http_warn` |
| Critical thresholds | `thresh_cpu_crit`, `thresh_ram_crit`, `thresh_disk_crit`, `thresh_http_crit`, `thresh_net_crit` |
| Scale-down thresholds | `thresh_cpu_low`, `thresh_ram_low`, `thresh_http_low` |

## 1. Imports & Configuration

In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

# ── Constants ─────────────────────────────────────────────────────────────────
SCRAPE_INTERVAL = 15          # seconds between Prometheus scrapes
MODELS_DIR      = './models'  # where .ubj model files are saved
DATA_DIR        = './data'
os.makedirs(MODELS_DIR, exist_ok=True)

# ── All 16 prediction targets ──────────────────────────────────────────────────
TARGET_WEIGHTS     = ['w_cpu', 'w_ram', 'w_io', 'w_energy']
TARGET_THRESH_WARN = ['thresh_cpu_warn', 'thresh_ram_warn', 'thresh_disk_warn', 'thresh_http_warn']
TARGET_THRESH_CRIT = ['thresh_cpu_crit', 'thresh_ram_crit', 'thresh_disk_crit', 'thresh_http_crit', 'thresh_net_crit']
TARGET_THRESH_LOW  = ['thresh_cpu_low', 'thresh_ram_low', 'thresh_http_low']
ALL_TARGETS        = TARGET_WEIGHTS + TARGET_THRESH_WARN + TARGET_THRESH_CRIT + TARGET_THRESH_LOW

print(f"XGBoost version : {xgb.__version__}")
print(f"Targets to predict ({len(ALL_TARGETS)} total): {ALL_TARGETS}")

## 2. Load Raw Data

The training dataset was captured from a live Proxmox cluster monitored by Prometheus.

- `1_prometheus_raw_metrics.csv` — raw Prometheus scrape data (cumulative counters from
  `node_exporter`, `scaphandre`, and `blackbox_exporter`).  
  Columns include `node_cpu_seconds_total_idle`, `node_memory_MemAvailable_bytes`,
  `node_disk_io_time_seconds_total`, `http_requests_total_all`, `http_requests_total_5xx`,
  `node_network_receive_packets_total`, `node_network_receive_drop_total`,
  `scaph_vm_power_microwatts`, plus `up`, `scrape_duration_seconds`, `instance`, `vm_name`, `vlan`.

- `2_cluster_manager_logs.csv` — pseudo-labels written by `main.py`'s `_record_ml_snapshot()`
  each time the cluster state is polled.  Contains the 16 target columns.

In [ ]:
df_metrics = pd.read_csv(f'{DATA_DIR}/1_prometheus_raw_metrics.csv')
df_logs    = pd.read_csv(f'{DATA_DIR}/2_cluster_manager_logs.csv')

print(f"Metrics shape : {df_metrics.shape}")
print(f"Logs shape    : {df_logs.shape}")
print(f"\nUnique instances : {df_metrics['instance'].nunique()}")
print(f"Time range       : {df_metrics['_time'].min()}  →  {df_metrics['_time'].max()}")

df_metrics.head(3)

In [ ]:
df_logs.head(3)

## 3. Feature Engineering

The raw CSV contains **cumulative counters** (e.g. `node_cpu_seconds_total_idle` keeps growing).  
XGBoost needs **instantaneous rates** — we compute per-instance deltas between consecutive scrapes.

> ⚠️ **Training vs. production asymmetry**  
> This `engineer_features()` function is used **only during training** (this notebook).  
> In production, `main.py`'s `_record_ml_snapshot()` derives the same features directly  
> from the live Proxmox API and Prometheus instant queries, writing pre-engineered values  
> to `metrics_latest.csv`.  `incremental_learning.py` reads that file directly without  
> calling `engineer_features()` again.

| Raw Column | Derived Feature | Formula |
|---|---|---|
| `node_cpu_seconds_total_idle` | `cpu_busy_pct` | `(1 - Δidle / scrape_interval) × 100` |
| `node_memory_MemAvailable_bytes` | `ram_usage_pct` | `(1 - avail/total) × 100` |
| `node_disk_io_time_seconds_total` | `io_util_pct` | `(Δio_time / scrape_interval) × 100` |
| `http_requests_total_5xx` / `_all` | `http_5xx_rate` | `Δ5xx / Δall` |
| `node_network_receive_drop_total` | `net_drop_rate` | `Δdrops / Δpackets` |
| `scaph_vm_power_microwatts` | `power_watts` | `microwatts / 1 000 000` |
| `vm_name` | `is_worker`, `is_master`, `is_monitor` | regex match |
| `vlan` | `vlan_enc` | `LabelEncoder` ordinal |

In [ ]:
# ── Sort so that per-instance diff() is chronologically correct ────────────────
df = df_metrics.sort_values(['instance', '_time']).reset_index(drop=True)

# ── 1. RAM usage % (instant gauge — no delta needed) ─────────────────────────
# ram_usage_pct = (used / total) * 100  where used = total - available
df['ram_usage_pct'] = (
    1 - df['node_memory_MemAvailable_bytes'] / df['node_memory_MemTotal_bytes']
) * 100

# ── 2. CPU busy % (diff of cumulative idle counter per instance) ───────────────
df['_cpu_idle_delta'] = df.groupby('instance')['node_cpu_seconds_total_idle'].diff()
df['_cpu_idle_delta'] = df['_cpu_idle_delta'].fillna(0).clip(lower=0)
idle_rate = (df['_cpu_idle_delta'] / SCRAPE_INTERVAL).clip(0, 1)
df['cpu_busy_pct'] = (1 - idle_rate) * 100

# ── 3. Disk IO utilisation % ──────────────────────────────────────────────────
df['_io_delta']  = df.groupby('instance')['node_disk_io_time_seconds_total'].diff().fillna(0).clip(lower=0)
df['io_util_pct'] = (df['_io_delta'] / SCRAPE_INTERVAL * 100).clip(0, 100)

# ── 4. HTTP 5xx error rate ────────────────────────────────────────────────────
df['_http_all_delta'] = df.groupby('instance')['http_requests_total_all'].diff().fillna(0).clip(lower=1)
df['_http_5xx_delta'] = df.groupby('instance')['http_requests_total_5xx'].diff().fillna(0).clip(lower=0)
df['http_5xx_rate']   = (df['_http_5xx_delta'] / df['_http_all_delta']).clip(0, 1)

# ── 5. Network packet drop rate ───────────────────────────────────────────────
df['_net_pkt_delta']  = df.groupby('instance')['node_network_receive_packets_total'].diff().fillna(0).clip(lower=1)
df['_net_drop_delta'] = df.groupby('instance')['node_network_receive_drop_total'].diff().fillna(0).clip(lower=0)
df['net_drop_rate']   = (df['_net_drop_delta'] / df['_net_pkt_delta']).clip(0, 1)

# ── 6. Power in watts ─────────────────────────────────────────────────────────
df['power_watts'] = df['scaph_vm_power_microwatts'] / 1_000_000

# ── 7. Role one-hot flags ─────────────────────────────────────────────────────
df['is_worker']  = df['vm_name'].str.contains('worker',              case=False).fillna(False).astype(int)
df['is_master']  = df['vm_name'].str.contains('master',              case=False).fillna(False).astype(int)
df['is_monitor'] = df['vm_name'].str.contains('monitoring|influx|snmp', case=False, regex=True).fillna(False).astype(int)

# ── 8. VLAN ordinal encoding ──────────────────────────────────────────────────
vlan_encoder = LabelEncoder()
df['vlan_enc'] = vlan_encoder.fit_transform(df['vlan'])
print("VLAN encoding:", dict(zip(vlan_encoder.classes_, vlan_encoder.transform(vlan_encoder.classes_))))

# ── Drop intermediate helper columns ──────────────────────────────────────────
helper_cols = ['_cpu_idle_delta', '_io_delta', '_http_all_delta', '_http_5xx_delta', '_net_pkt_delta', '_net_drop_delta']
df.drop(columns=helper_cols, inplace=True)

print(f"\nFeature engineering complete. Shape: {df.shape}")
df[['instance','cpu_busy_pct','ram_usage_pct','io_util_pct','http_5xx_rate','net_drop_rate','power_watts']].describe()

## 4. Merge Metrics with Manager Logs (targets)

In [ ]:
# Inner join on time + instance + vm_name
# Drops monitoring-only rows and the first scrape per instance (no delta available)
df_merged = pd.merge(
    df, df_logs,
    on=['_time', 'instance', 'vm_name'],
    how='inner'
)

print(f"Merged shape: {df_merged.shape}")
print(f"Columns     : {df_merged.columns.tolist()}")

## 5. Define X (Features) and Y (Targets)

### What goes in X and why

| Feature | Why it matters |
|---|---|
| `up` | VM reachability — unreachable nodes should get zero weight |
| `scrape_duration_seconds` | High latency → CPU bottleneck indicator |
| `cpu_busy_pct` | Primary driver of `w_cpu` and `thresh_cpu_*` |
| `ram_usage_pct` | Primary driver of `w_ram` and `thresh_ram_*` |
| `io_util_pct` | Primary driver of `w_io` and `thresh_disk_*` |
| `http_5xx_rate` | Error rate — drives `thresh_http_*` |
| `net_drop_rate` | Network quality — drives `thresh_net_crit` |
| `power_watts` | Energy consumption — driver of `w_energy` |
| `is_worker`, `is_master`, `is_monitor` | VM role — different roles have different normal load profiles |
| `vlan_enc` | Network segment — VLANs may have different traffic patterns |

### Feature engineering contract

These 12 features are computed **identically** in three places:
1. This notebook (via `engineer_features()` from raw Prometheus counters)
2. `main.py`'s `_record_ml_snapshot()` (from live Proxmox API + Prometheus instant queries)
3. `incremental_learning.py` (reads pre-engineered values directly from `metrics_latest.csv`)

Any change to the feature list must be propagated to all three.

In [ ]:
FEATURE_COLS = [
    'up',
    'scrape_duration_seconds',
    'cpu_busy_pct',
    'ram_usage_pct',
    'io_util_pct',
    'http_5xx_rate',
    'net_drop_rate',
    'power_watts',
    'is_worker',
    'is_master',
    'is_monitor',
    'vlan_enc',
]

# Keep only rows where all features AND all targets are non-null
df_model = df_merged[FEATURE_COLS + ALL_TARGETS].dropna()

X = df_model[FEATURE_COLS]
y = df_model[ALL_TARGETS]

print(f"Final dataset: {X.shape[0]} samples × {X.shape[1]} features")
print(f"Targets      : {ALL_TARGETS}")
print(f"\nFeature stats:")
X.describe().round(3)

In [ ]:
# Quick look at target distributions
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()

colors = cm.tab20.colors
for i, target in enumerate(ALL_TARGETS):
    axes[i].hist(y[target], bins=40, color=colors[i % len(colors)], alpha=0.75, edgecolor='white')
    axes[i].set_title(target, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')
    axes[i].grid(axis='y', alpha=0.3)

for j in range(len(ALL_TARGETS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Target Variable Distributions', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('./models/target_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Train/Test Split

> ⚠️ **Important**: `stratify=` is **only for classification** (it balances class labels).  
> For multi-output **regression**, use a plain `shuffle=True` split.

In [ ]:
# 80% train / 20% test — NO stratify (regression, not classification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test : {X_test.shape[0]} samples")

## 7. Train One XGBRegressor per Target

Each model is saved as `models/<target>.ubj` — the native XGBoost binary format.  
These files are loaded at startup by both **`main.py`** (local fallback via `_load_ml_models()`)  
and **`ml_service.py`** (via `_load_models()`), and reloaded after each incremental update.

One `XGBRegressor` is trained per target (16 total). Using `early_stopping_rounds=30`  
against the test set prevents overfitting and keeps models compact.

In [ ]:
XGB_PARAMS = dict(
    n_estimators     = 300,
    max_depth        = 6,
    learning_rate    = 0.05,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    min_child_weight = 3,
    tree_method      = 'hist',
    objective        = 'reg:squarederror',
    random_state     = 42,
    early_stopping_rounds = 30,
    eval_metric      = 'mae',
)

models = {}

for target in ALL_TARGETS:
    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(
        X_train, y_train[target],
        eval_set=[(X_test, y_test[target])],
        verbose=False,
    )
    models[target] = model
    model.save_model(f'{MODELS_DIR}/{target}.ubj')
    best = model.best_iteration
    print(f"  [{target:20s}]  best_iteration={best:3d}  "
          f"MAE_train={mean_absolute_error(y_train[target], model.predict(X_train)):.4f}  "
          f"MAE_test={mean_absolute_error(y_test[target], model.predict(X_test)):.4f}")

print(f"\n✓  {len(ALL_TARGETS)} models saved to '{MODELS_DIR}/'")

## 8. Evaluation

In [ ]:
print(f"{'Target':20s}  {'MAE':>8s}  {'R²':>7s}")
print("-" * 40)
results = {}
for target in ALL_TARGETS:
    preds = models[target].predict(X_test)
    mae   = mean_absolute_error(y_test[target], preds)
    r2    = r2_score(y_test[target], preds)
    results[target] = {'mae': mae, 'r2': r2}
    quality = '✓' if r2 > 0.85 else ('~' if r2 > 0.6 else '✗')
    print(f"{target:20s}  {mae:8.4f}  {r2:7.4f}  {quality}")

In [ ]:
# Feature importances for the 4 weight models
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, target in enumerate(TARGET_WEIGHTS):
    importances = pd.Series(
        models[target].feature_importances_,
        index=FEATURE_COLS
    ).sort_values(ascending=True)
    importances.plot(kind='barh', ax=axes[i], color='#4C72B0', alpha=0.8)
    axes[i].set_title(f'Feature Importance — {target}', fontweight='bold')
    axes[i].set_xlabel('Importance Score')
    axes[i].grid(axis='x', alpha=0.3)

plt.suptitle('XGBoost Feature Importances (Fitness Weights)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('./models/feature_importances.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Incremental Learning

XGBoost supports **warm-start incremental training** via the `xgb_model=` parameter of `xgb.train()`.  
This **appends new trees** on top of the existing booster — old knowledge is preserved.

```
Existing model (300 trees)  +  New 5-min batch  →  Updated model (320 trees)
```

### Training vs. production data contract

The `engineer_features()` helper below re-implements the training pipeline for **notebook demos**  
and **offline re-training** — it processes raw Prometheus counter CSVs.  

In **production** the `incremental_update()` function defined in `incremental_learning.py`  
is called by `main.py` every 5 minutes. It receives *pre-engineered* CSVs written by  
`main.py`'s `_record_ml_snapshot()` and skips `engineer_features()` entirely.

### XGBoost 2.x compatibility note

> `verbose_eval` is **not** a valid `xgb.train()` parameter in XGBoost ≥ 2.x.  
> Use `"verbosity": 0` inside the params dict instead (as shown below).

In [ ]:
def engineer_features(df_raw: pd.DataFrame, vlan_enc: LabelEncoder) -> pd.DataFrame:
    """
    Re-usable feature engineering pipeline for the training pipeline.
    Input  : raw Prometheus metrics DataFrame
             (same schema as 1_prometheus_raw_metrics.csv)
    Output : DataFrame with FEATURE_COLS ready for XGBoost.

    NOTE: This function is used ONLY during training / offline demo.
    Production incremental updates use pre-engineered data from
    main.py's _record_ml_snapshot() — see incremental_learning.py.
    """
    df = df_raw.sort_values(['instance', '_time']).reset_index(drop=True).copy()

    # RAM: (used / total) * 100 — 'used' = total - available
    df['ram_usage_pct']  = (1 - df['node_memory_MemAvailable_bytes'] / df['node_memory_MemTotal_bytes']) * 100

    cpu_idle_d           = df.groupby('instance')['node_cpu_seconds_total_idle'].diff().fillna(0).clip(lower=0)
    df['cpu_busy_pct']   = (1 - (cpu_idle_d / SCRAPE_INTERVAL).clip(0, 1)) * 100

    io_d                 = df.groupby('instance')['node_disk_io_time_seconds_total'].diff().fillna(0).clip(lower=0)
    df['io_util_pct']    = (io_d / SCRAPE_INTERVAL * 100).clip(0, 100)

    http_all_d           = df.groupby('instance')['http_requests_total_all'].diff().fillna(0).clip(lower=1)
    http_5xx_d           = df.groupby('instance')['http_requests_total_5xx'].diff().fillna(0).clip(lower=0)
    df['http_5xx_rate']  = (http_5xx_d / http_all_d).clip(0, 1)

    pkt_d                = df.groupby('instance')['node_network_receive_packets_total'].diff().fillna(0).clip(lower=1)
    drop_d               = df.groupby('instance')['node_network_receive_drop_total'].diff().fillna(0).clip(lower=0)
    df['net_drop_rate']  = (drop_d / pkt_d).clip(0, 1)

    df['power_watts']    = df['scaph_vm_power_microwatts'] / 1_000_000
    df['is_worker']      = df['vm_name'].str.contains('worker', case=False).astype(int)
    df['is_master']      = df['vm_name'].str.contains('master', case=False).astype(int)
    df['is_monitor']     = df['vm_name'].str.contains('monitoring|influx|snmp', case=False, regex=True).astype(int)

    # Handle unseen VLAN labels gracefully
    known_vlans = set(vlan_enc.classes_)
    df['vlan_safe'] = df['vlan'].where(df['vlan'].isin(known_vlans), other=vlan_enc.classes_[0])
    df['vlan_enc']  = vlan_enc.transform(df['vlan_safe'])

    return df

print("engineer_features() defined ✓")

In [ ]:
def incremental_update(
    new_metrics_csv: str,
    new_logs_csv: str,
    n_new_trees: int = 20,
    models_dir: str = MODELS_DIR,
    from_raw_prometheus: bool = True,
) -> dict:
    """
    Add `n_new_trees` boosting rounds to each model using a fresh data batch.

    Parameters
    ----------
    new_metrics_csv      : path to the metrics CSV batch
    new_logs_csv         : path to the logs / pseudo-labels CSV batch
    n_new_trees          : number of new trees to add per model (default 20)
    models_dir           : directory where .ubj files are stored
    from_raw_prometheus  : if True (notebook/demo), run engineer_features() on the
                           raw Prometheus counter CSV first.
                           If False (production), the metrics CSV already contains
                           pre-engineered FEATURE_COLS — skip engineer_features().

    Returns
    -------
    dict  { target -> {mae_before, mae_after, delta} }

    Production note
    ---------------
    In production, main.py calls incremental_learning.incremental_update() directly
    (always with pre-engineered data, i.e. from_raw_prometheus=False).  This notebook
    version exists for offline demos and re-training experiments.

    XGBoost 2.x note
    ----------------
    verbose_eval is NOT a valid xgb.train() parameter in XGBoost >= 2.x.
    Use 'verbosity': 0 inside the params dict instead.
    """
    df_m_raw = pd.read_csv(new_metrics_csv)
    df_l_raw = pd.read_csv(new_logs_csv)

    if from_raw_prometheus:
        # Training / demo path: raw Prometheus counters → engineer features
        df_m = engineer_features(df_m_raw, vlan_encoder)
    else:
        # Production path: features already computed by main.py
        df_m = df_m_raw

    df_new = pd.merge(df_m, df_l_raw, on=['_time', 'instance', 'vm_name'], how='inner')
    df_new = df_new[FEATURE_COLS + ALL_TARGETS].dropna()

    if len(df_new) < 10:
        print("⚠ Not enough new samples for update (need ≥ 10).")
        return {}

    X_new = df_new[FEATURE_COLS]
    y_new = df_new[ALL_TARGETS]

    update_report = {}

    for target in ALL_TARGETS:
        # 1. Load existing booster from disk
        existing_booster = xgb.Booster()
        existing_booster.load_model(f'{models_dir}/{target}.ubj')

        # MAE before update
        d_new        = xgb.DMatrix(X_new, label=y_new[target], feature_names=FEATURE_COLS)
        preds_before = existing_booster.predict(d_new)
        mae_before   = mean_absolute_error(y_new[target], preds_before)

        # 2. Append new trees — xgb_model= preserves existing trees
        # NOTE: 'verbosity': 0 silences output; verbose_eval is not valid in XGBoost >= 2.x
        params = {
            'max_depth'        : 6,
            'learning_rate'    : 0.05,
            'subsample'        : 0.8,
            'colsample_bytree' : 0.8,
            'min_child_weight' : 3,
            'tree_method'      : 'hist',
            'objective'        : 'reg:squarederror',
            'eval_metric'      : 'mae',
            'seed'             : 42,
            'verbosity'        : 0,   # ← correct way to silence in XGBoost >= 2.x
        }
        updated_booster = xgb.train(
            params,
            d_new,
            num_boost_round = n_new_trees,
            xgb_model       = existing_booster,   # ← incremental learning
        )

        # MAE after update
        preds_after = updated_booster.predict(d_new)
        mae_after   = mean_absolute_error(y_new[target], preds_after)

        # 3. Save updated model (overwrites previous)
        updated_booster.save_model(f'{models_dir}/{target}.ubj')

        update_report[target] = {
            'mae_before': round(mae_before, 5),
            'mae_after' : round(mae_after,  5),
            'delta'     : round(mae_after - mae_before, 5),
        }
        improvement = '↓' if mae_after < mae_before else '↑'
        print(f"  [{target:20s}]  MAE {mae_before:.5f} → {mae_after:.5f}  {improvement}")

    print(f"\n✓  Incremental update complete (+{n_new_trees} trees per model)")
    return update_report

print("incremental_update() defined ✓")

In [ ]:
# ── Demo: simulate an incremental update using the test split ─────────────────
#
# In production, main.py writes pre-engineered features directly to
# DATA_BUFFER/metrics_latest.csv and DATA_BUFFER/logs_latest.csv every 10 s,
# then incremental_learning.incremental_update() is called with
# from_raw_prometheus=False (no engineer_features() step).
#
# Here we reconstruct raw-counter rows from the merged dataset so that the
# notebook's engineer_features() can be exercised end-to-end.

demo_metrics_path = f'{DATA_DIR}/demo_new_metrics.csv'
demo_logs_path    = f'{DATA_DIR}/demo_new_logs.csv'

demo_rows = df_merged.loc[X_test.index].copy()
# Write raw-format metrics (includes original Prometheus counter columns)
demo_rows[df_metrics.columns.tolist()].to_csv(demo_metrics_path, index=False)
# Write logs (pseudo-labels)
demo_rows[df_logs.columns.tolist()].to_csv(demo_logs_path, index=False)

print("Running incremental update demo on test batch (raw Prometheus counters)...")
report = incremental_update(
    demo_metrics_path, demo_logs_path,
    n_new_trees=20,
    from_raw_prometheus=True,   # notebook demo: engineer features from raw counters
)

## 10. Prediction Function

This utility function can be used in notebooks and scripts for single-VM predictions.  
In production, predictions are served by **`ml_service.py`** (via its `/sync` endpoint)  
and **`main.py`** (via its local `/predict-config/{instance_id}` fallback endpoint).

In [ ]:
def load_models(models_dir: str = MODELS_DIR) -> dict[str, xgb.Booster]:
    """
    Load all 16 trained boosters from disk.
    Called once at startup in main.py (via _load_ml_models()) and in
    ml_service.py (via _load_models()).
    """
    loaded = {}
    for target in ALL_TARGETS:
        b = xgb.Booster()
        b.load_model(f'{models_dir}/{target}.ubj')
        loaded[target] = b
    print(f"Loaded {len(loaded)} models from '{models_dir}'")
    return loaded


def predict_config(current_metrics: dict, boosters: dict) -> dict:
    """
    Predict optimal weights and thresholds for one VM snapshot.

    Parameters
    ----------
    current_metrics : dict with keys matching FEATURE_COLS.
                      ram_usage_pct = (used_bytes / max_bytes) * 100
                      (used_bytes = mem field from Proxmox, max_bytes = maxmem field)
    boosters        : dict[target_name → xgb.Booster], from load_models()

    Returns
    -------
    dict with predicted and post-processed weights + thresholds
    """
    X_input = xgb.DMatrix(
        pd.DataFrame([current_metrics])[FEATURE_COLS],
        feature_names=FEATURE_COLS
    )

    raw = {target: float(boosters[target].predict(X_input)[0]) for target in ALL_TARGETS}

    # Post-processing 1: normalise weights to sum exactly to 1.0
    w_sum = sum(max(0.0, raw[t]) for t in TARGET_WEIGHTS)
    if w_sum > 0:
        for t in TARGET_WEIGHTS:
            raw[t] = max(0.0, raw[t]) / w_sum
    else:
        for t in TARGET_WEIGHTS:
            raw[t] = 0.25

    # Post-processing 2: clamp thresholds to safe operating ranges
    raw['thresh_cpu_warn']  = float(np.clip(raw['thresh_cpu_warn'],  50.0, 95.0))
    raw['thresh_ram_warn']  = float(np.clip(raw['thresh_ram_warn'],  55.0, 95.0))
    raw['thresh_disk_warn'] = float(np.clip(raw['thresh_disk_warn'], 10.0, 95.0))
    raw['thresh_http_warn'] = float(np.clip(raw['thresh_http_warn'],  0.1,  5.0))

    raw['thresh_cpu_crit']  = float(np.clip(raw['thresh_cpu_crit'],  50.0, 95.0))
    raw['thresh_ram_crit']  = float(np.clip(raw['thresh_ram_crit'],  55.0, 95.0))
    raw['thresh_disk_crit'] = float(np.clip(raw['thresh_disk_crit'], 10.0, 95.0))
    raw['thresh_http_crit'] = float(np.clip(raw['thresh_http_crit'],  0.1,  5.0))
    raw['thresh_net_crit']  = float(np.clip(raw['thresh_net_crit'],   0.1,  5.0))

    raw['thresh_cpu_low']   = float(np.clip(raw['thresh_cpu_low'],  10.0, 50.0))
    raw['thresh_ram_low']   = float(np.clip(raw['thresh_ram_low'],  10.0, 50.0))
    raw['thresh_http_low']  = float(np.clip(raw['thresh_http_low'],  0.1,  1.0))

    return raw


# ── Smoke test ─────────────────────────────────────────────────────────────────
boosters = load_models()

sample_input = {
    'up'                      : 1.0,
    'scrape_duration_seconds' : 0.12,
    'cpu_busy_pct'            : 78.5,
    # ram_usage_pct = (used / maxmem) * 100  — NOT (1 - avail/total) in the Proxmox context
    # Example: 6.5 GB used out of 10 GB = 65.0%
    'ram_usage_pct'           : 65.0,
    'io_util_pct'             : 12.0,
    'http_5xx_rate'           : 0.03,
    'net_drop_rate'           : 0.001,
    'power_watts'             : 95.0,
    'is_worker'               : 1,
    'is_master'               : 0,
    'is_monitor'              : 0,
    'vlan_enc'                : 1,
}

result = predict_config(sample_input, boosters)
print("\nSample prediction:")
for k, v in result.items():
    print(f"  {k:20s} = {v:.4f}")
print(f"\n  Weight sum check: {sum(result[t] for t in TARGET_WEIGHTS):.6f}  (should be 1.0)")

## 11. Integration Architecture

This notebook trains the initial models.  The production system uses three separate services.

```
┌──────────────────────────────────────────────────────────────────┐
│  main.py  (Cluster Manager — FastAPI, port 8000)                 │
│                                                                  │
│  • poll_cluster() every 10 s                                     │
│      └─ _record_ml_snapshot() → metrics_latest.csv              │
│                                   logs_latest.csv               │
│  • ml_incremental_update_job() every 5 min                       │
│      ├─ incremental_learning.incremental_update()                │
│      │   (pre-engineered CSVs — no engineer_features() call)     │
│      ├─ POST ml_service:8001/reload-models                       │
│      └─ POST ml_service:8001/sync  →  _ml_config refresh        │
│  • _sync_with_ml_service() every 1 min                           │
│  • hybrid_de_woa_provisioning() uses W_CPU/W_RAM/W_IO/W_E        │
│  • check_scaledown() uses thresh_*_low from _ml_config           │
│  • POST app.py:5000/add-target  (on VM clone)                    │
│  • POST app.py:5000/remove-target  (on VM delete)                │
└──────────────────────────────────────────────────────────────────┘
           ▲ POST /migrate-vm
           │
┌──────────────────────────────────────────────────────────────────┐
│  app.py  (Prometheus SD API — FastAPI, port 5000)                │
│                                                                  │
│  • GET /targets/{job}  →  Prometheus HTTP SD                     │
│  • POST /add-target    (called by main.py on clone)              │
│  • POST /remove-target (called by main.py on delete)             │
│  • POST /update-thresholds → rewrites vm_scaling.yml             │
│      └─ POST Prometheus/-/reload                                 │
│  • PM health monitor thread                                      │
│      └─ POST main.py:8000/migrate-vm  (on PM failure)            │
└──────────────────────────────────────────────────────────────────┘
                          ▲ POST /update-thresholds
                          │
┌──────────────────────────────────────────────────────────────────┐
│  ml_service.py  (XGBoost API — FastAPI, port 8001)               │
│                                                                  │
│  • POST /sync  →  predict config from VM batch                   │
│  • POST /reload-models  →  reload .ubj files from disk           │
│  • GET  /config  →  last predicted config                        │
└──────────────────────────────────────────────────────────────────┘
```

### Key design decisions

- **ml_service.py is stateless between syncs** — it does not write to disk; only
  `incremental_learning.py` (called from `main.py`) modifies the `.ubj` files.
- **main.py holds _ml_config** — weights and thresholds are applied there for
  the DE-WOA algorithm and scale-down engine.  ml_service.py only predicts.
- **app.py owns Prometheus config** — it is the only component that modifies
  `vm_scaling.yml` and triggers a Prometheus rules reload.

## 12. Save Artefacts

In [ ]:
import json

# Save VLAN encoder so main.py and ml_service.py can rebuild it at startup
joblib.dump(vlan_encoder, f'{MODELS_DIR}/vlan_encoder.joblib')

# Save feature column order (critical — must match main.py and ml_service.py exactly)
with open(f'{MODELS_DIR}/feature_cols.json', 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)

# Save evaluation results
pd.DataFrame(results).T.to_csv(f'{MODELS_DIR}/evaluation_results.csv')

print("Saved artefacts:")
for fname in sorted(os.listdir(MODELS_DIR)):
    fpath = os.path.join(MODELS_DIR, fname)
    print(f"  {fname:40s}  {os.path.getsize(fpath) / 1024:.1f} KB")